In [1]:
# Preprocess CESM2 surface ozone data
# Before adding a new scenario, check the data format

In [2]:
# WANT: Hourly surface ozone in ppb

In [3]:
import os
import numpy as np
import xarray as xr
from utils.utils import get_scenario_config, load_file_list, molmol_to_ppb

In [4]:
# Load file and convert units
def load_ozone_ppb(ds, model_name):
    config = SCENARIO_CONFIG[model_name]
    var_name = config["var"]
    converter = config["convert"]

    data = ds[var_name]
    converted = converter(data)
    converted.attrs["units"] = "ppb"
    return converted


# === Return frequency string e.g '3-hourly' for xarray with time coord ===
def time_frequency(data):
    step = data.time.diff("time").median()
    # convert to hours
    step_hours = step / np.timedelta64(1, "h")
    # handle whole days if nicer
    if step_hours % 24 == 0:
        return f"{int(step_hours/24)}-daily"
    else:
        return f"{int(step_hours)}-hourly"

In [5]:
# Map scenario -> variable name + conversion
# This may be different across scenarios for other models - CHECK
SCENARIO_CONFIG = {
    "SSP245_G6": {
        "var": "O3_SRF",
        "convert": molmol_to_ppb},
    "G6-1.5K": {
        "var": "O3_SRF",
        "convert": molmol_to_ppb},
    "hist": {
        "var": "O3_SRF",
        "convert": molmol_to_ppb},
}

In [6]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "hist"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

FILE_DIR = f"/glade/work/awells/air_quality/{model}/ozone/file_paths/"
# Located in SCRATCH to save space
SAVE_DIR = f"/glade/derecho/scratch/awells/air_quality/{model}/ozone/hourly_o3/"

# === MAIN LOOP ===

for ens_num in ensemble_members:
    print(f"Processing {scenario}, ensemble {ens_num:02d}")
    file_list = load_file_list(FILE_DIR, f"file_list_{scenario}_{ens_num}.json")

    for f in file_list:
        if not os.path.exists(f):
            raise ValueError(f"Missing: {f}")

        print(f"Reading {os.path.basename(f)}")
        da = load_ozone_ppb(xr.open_dataset(f), scenario)

        if time_frequency(da) != "1-hourly":
            raise ValueError("Input data is not hourly, "
                             "convert data to hourly before continuing")

        # Create date stamp for file saving
        start_time = da["time"][0].item().strftime("%Y%m%d")
        end_time = da["time"][-1].item().strftime("%Y%m%d")
        dates = f"{start_time}-{end_time}"

        description = ("Processed hourly surface ozone "
                       "- scripts by A.F. Wells (2025)")
        da.attrs["description"] = description
        da.attrs["ensemble_number"] = ens_num
        da.attrs["scenario"] = scenario
        da.attrs["model"] = model

        out_file = f"Hourly_surface_o3_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)
        print(f"Saving to {out_path}")
        da.to_netcdf(out_path)

        del da

print("All processing complete.")